In [17]:
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
response = llm.invoke("Say 'Stage 5 LLM connected successfully' and nothing else.")
print(response.content)

Stage 5 LLM connected successfully


In [11]:
import json
import numpy as np
import pandas as pd
import faiss
import chromadb
from sklearn.preprocessing import normalize
from langchain_groq import ChatGroq
from langchain.tools import Tool
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

In [12]:

# ── Load all Stage 4 outputs ──────────────────────────────────────────────────
product_vector_df = pd.read_csv("models/absa/product_aspect_vectors.csv")
product_ids = product_vector_df["product_id"].tolist()
feature_cols = [c for c in product_vector_df.columns if c != "product_id"]
vectors = product_vector_df[feature_cols].values.astype("float32")
vectors_normalized = normalize(vectors).astype("float32")

# Rebuild FAISS index
dimension = vectors_normalized.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(vectors_normalized)

# Load ChromaDB
chroma_client = chromadb.PersistentClient(path="models/stage4/chroma_db")
collection = chroma_client.get_or_create_collection(name="product_reviews")

# Load taxonomy map
with open("models/absa/aspect_taxonomy_map.json", "r") as f:
    aspect_to_taxonomy = json.load(f)

print("✅ All Stage 4 outputs loaded")

✅ All Stage 4 outputs loaded


In [13]:

# ── Define 4 Tools ────────────────────────────────────────────────────────────

# Tool 1: Get product aspect vector
def get_product_profile(product_id: str) -> str:
    product_id = product_id.strip()
    if product_id not in product_ids:
        return f"Product {product_id} not found."
    idx = product_ids.index(product_id)
    vec = vectors[idx]
    profile = {feature_cols[i]: round(float(vec[i]), 3) for i in range(len(feature_cols))}
    # Sort by score descending
    profile = dict(sorted(profile.items(), key=lambda x: x[1], reverse=True))
    return json.dumps(profile)

# Tool 2: Find similar products
def similar_products(product_id: str) -> str:
    product_id = product_id.strip()
    if product_id not in product_ids:
        return f"Product {product_id} not found."
    idx = product_ids.index(product_id)
    query_vec = vectors_normalized[idx].reshape(1, -1)
    scores, indices = index.search(query_vec, 6)
    results = []
    for score, i in zip(scores[0], indices[0]):
        if i == idx:
            continue
        results.append(f"{product_ids[i]} (similarity: {round(float(score), 3)})")
    return "\n".join(results[:5])

# Tool 3: Gap-fill recommender
def gap_fill_recommender(product_id: str) -> str:
    product_id = product_id.strip()
    if product_id not in product_ids:
        return f"Product {product_id} not found."
    idx = product_ids.index(product_id)
    vec = vectors[idx]
    weak_aspects = [feature_cols[i] for i, s in enumerate(vec) if s < -0.1]
    if not weak_aspects:
        return f"Product {product_id} has no weak aspects — it scores well on everything."
    gap_scores = []
    for i, pid in enumerate(product_ids):
        if pid == product_id:
            continue
        other_vec = vectors[i]
        gap_score = np.mean([other_vec[feature_cols.index(a)] for a in weak_aspects])
        gap_scores.append((pid, round(float(gap_score), 4)))
    gap_scores.sort(key=lambda x: x[1], reverse=True)
    result = f"Weak aspects: {weak_aspects}\nBetter alternatives:\n"
    result += "\n".join([f"{pid} (gap score: {score})" for pid, score in gap_scores[:5]])
    return result

# Tool 4: Fetch real reviews from ChromaDB
def fetch_reviews(product_id: str) -> str:
    product_id = product_id.strip()
    results = collection.query(
        query_texts=[f"review about {product_id}"],
        n_results=3,
        where={"product_id": product_id}
    )
    if not results["documents"][0]:
        return f"No reviews found for {product_id}."
    output = ""
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        output += f"[Score: {meta['score']}] {meta['summary']}: {doc[:200]}\n\n"
    return output

In [14]:

tools = [
    Tool(name="get_product_profile",
         func=get_product_profile,
         description="Get the aspect sentiment vector for a product. Input: product_id (e.g. B001E4KFG0)"),
    Tool(name="similar_products",
         func=similar_products,
         description="Find products most similar to a given product. Input: product_id"),
    Tool(name="gap_fill_recommender",
         func=gap_fill_recommender,
         description="Find products that are better in the weak aspects of a given product. Input: product_id"),
    Tool(name="fetch_reviews",
         func=fetch_reviews,
         description="Fetch real customer review snippets for a product from the database. Input: product_id"),
]

print("✅ Tools registered")

✅ Tools registered


In [15]:
# ── Build Agent ───────────────────────────────────────────────────────────────
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Pull standard ReAct prompt from LangChain hub
prompt = hub.pull("hwchase17/react")

agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,          # shows reasoning steps
    max_iterations=8,
    handle_parsing_errors=True
)

print("✅ Agent ready")

# ── Test 1: Simple product profile ───────────────────────────────────────────
response = agent_executor.invoke({
    "input": "What are the strengths and weaknesses of product B000LQOCH0 based on customer reviews?"
})

print("\n── Final Answer ──")
print(response["output"])

✅ Agent ready


> Entering new AgentExecutor chain...
To determine the strengths and weaknesses of product B000LQOCH0 based on customer reviews, I need to analyze the sentiment of customer reviews for this product. 

Action: fetch_reviews
Action Input: B000LQOCH0[Score: 4.0] "Delight" says it all: This is a confection that has been around a few centuries.  It is a light, pillowy citrus gelatin with nuts - in this case Filberts. And it is cut into tiny squares and then liberally coated with powd

Thought: The observation from the fetch_reviews action provides some insight into the customer's opinion of the product, but it's just a single review snippet. To get a more comprehensive understanding of the strengths and weaknesses of the product, I need to analyze the aspect sentiment vector for the product.

Action: get_product_profile
Action Input: B000LQOCH0{"availability_location": 0.0, "beverage": 0.0, "brand_quality": 0.0, "food_ingredients": 0.0, "freshness_aroma": 0.0, "nutrition_ing

In [16]:
# ── Test 2: Recommendation query ─────────────────────────────────────────────
response = agent_executor.invoke({
    "input": "I'm not happy with product B000LQOCH0. Can you recommend a better alternative and explain why it's better?"
})

print("\n── Final Answer ──")
print(response["output"])



> Entering new AgentExecutor chain...
Thought: To recommend a better alternative to product B000LQOCH0, I first need to understand the aspects where this product is weak. This can be achieved by analyzing the sentiment of customer reviews for this product, which can give insights into its strengths and weaknesses.

Action: get_product_profile
Action Input: B000LQOCH0{"availability_location": 0.0, "beverage": 0.0, "brand_quality": 0.0, "food_ingredients": 0.0, "freshness_aroma": 0.0, "nutrition_ingredients": 0.0, "packaging_delivery": 0.0, "packaging_format": 0.0, "price_value": 0.0, "quantity_size": 0.0, "taste_flavor": 0.0, "food_type": -1.0, "target_user": -1.0}From the product profile, I can see that the product B000LQOCH0 has negative sentiment towards "food_type" and "target_user". This suggests that customers are not satisfied with the type of food this product is and the target user it is intended for.

Action: gap_fill_recommender
Action Input: B000LQOCH0Weak aspects: ['food_